# The Scaler That Saw the Test Set

**Topic:** Preprocessing and leakage &nbsp;·&nbsp; **Difficulty:** Medium

A teammate reports 84% cross-validated accuracy. In production the model gets
83%: a small but systematic optimism, and a warning sign, because it appears in
every report the team produces. Nothing about the model is wrong: the
standardizer was fitted on the whole dataset before the split, so every fold's
"held-out" data had already contributed its mean and variance to the transform.

Measure the gap yourself. Fit the same classifier two ways and watch the leaky
number float free of reality.

## The data (created in the next cell)

- `X` `(600, 300)` and `y` `(600,)`, a **wide** dataset: 300 features, 600 rows.
- Only the first 5 features carry any signal, and the other 295 are pure noise.
- Wide-and-noisy is exactly where leakage does the most damage.

## Your task

Using **NumPy and sklearn**, with `KFold(n_splits=5, shuffle=True, random_state=0)`
and `LogisticRegression(max_iter=2000)`:

1. `leaky_acc`: standardize **all** of `X` once with the full-data mean and
   standard deviation, then run 5-fold cross-validation on the transformed data.
2. `honest_acc`: inside each fold, fit the standardizer on the **training rows
   only**, apply it to both parts, then fit and score.
3. Set `answers` to a dict with `"leaky_acc"`, `"honest_acc"`, and `"gap"` =
   `leaky_acc - honest_acc`.

## Evaluation

Both accuracies are compared to the reference to a relative tolerance of 1e-6,
and the evaluation cell checks the leaky estimate is the optimistic one.

In [ ]:
import numpy as np

rng = np.random.default_rng(101)
n, d = 600, 300
y = rng.integers(0, 2, n)
X = rng.normal(0, 1, (n, d))
X[:, :5] += (y[:, None] - 0.5) * 1.1        # only 5 informative features
print(f"X {X.shape}, informative features: 5 of {d}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("leaky_acc", [], 0.8383333333333333, -0.14036886908615512, 1e-06),
    ("honest_acc", [], 0.8366666666666667, -0.14008980572813096, 1e-06),
    ("gap", [], 0.0016666666666665941, -0.0002790633580241531, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "gap" in answers:
    _g = answers["gap"] > 0
    print(f"  {'leaky is optimistic':<24} {'ok' if _g else 'wrong'}  (gap = {answers['gap']:+.4f})")
    _ok = _ok and _g

print("=" * 56)
if _ok:
    print("PASSED. Nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

The leak is small per feature but there are 300 of them, and the classifier is
free to exploit any test-set structure the shared scaling smuggled in. The honest
version is the only one that answers "how will this do on data I have not seen",
which is the only question that matters. In practice, put the scaler in a
`Pipeline` so it cannot be fitted outside the fold by accident.

```python
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=0)

Xall = (X - X.mean(axis=0)) / X.std(axis=0)
leaky = []
for tr, te in kf.split(Xall):
    m = LogisticRegression(max_iter=2000).fit(Xall[tr], y[tr])
    leaky.append((m.predict(Xall[te]) == y[te]).mean())

honest = []
for tr, te in kf.split(X):
    mu, sd = X[tr].mean(axis=0), X[tr].std(axis=0)
    m = LogisticRegression(max_iter=2000).fit((X[tr] - mu) / sd, y[tr])
    honest.append((m.predict((X[te] - mu) / sd) == y[te]).mean())

leaky_acc = float(np.mean(leaky))
honest_acc = float(np.mean(honest))
answers = {"leaky_acc": leaky_acc, "honest_acc": honest_acc,
           "gap": leaky_acc - honest_acc}
```